### STEP 1: Environment Setup + GPU Verification

In this step, we set up the required libraries and verify that a GPU is available in Google Colab.
Using a GPU is important because training language models (even small ones) on CPU will be extremely slow.

In [2]:
# Install required libraries
# torch: deep learning framework
# transformers: loads pretrained language models/tokenizers
# datasets: loads the Anthropic/hh-rlhf dataset
# numpy: numerical operations

!pip install -q torch transformers datasets numpy


# Import PyTorch
import torch


# Check whether GPU is available
print("Is GPU available?:", torch.cuda.is_available())


# Print GPU name if available
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected.")
    print("Go to: Runtime > Change runtime type > Select GPU")


# Set device for later training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Is GPU available?: True
GPU Name: Tesla T4
Using device: cuda


### Step 2: Load Model and Tokenizer

This step loads the approved low-compute base language model and tokenizer. We also configure right padding and reuse the EOS token as the PAD token if needed.

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Low-compute approved model for Google Colab
MODEL_NAME = "HuggingFaceTB/SmolLM2-360M"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Use right padding as required for the assignment
tokenizer.padding_side = "right"

# If tokenizer has no pad token, reuse EOS token as pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load pretrained causal language model
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Move model to GPU
model = model.to(device)

# Print required report information
print("Checkpoint:", MODEL_NAME)
print("Tokenizer vocab size:", len(tokenizer))
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)
print("Model device:", next(model.parameters()).device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Checkpoint: HuggingFaceTB/SmolLM2-360M
Tokenizer vocab size: 49152
Pad token: <|endoftext|>
EOS token: <|endoftext|>
Model device: cuda:0


### Step 3: Forward Pass and Logits Shape Check

This step verifies that the loaded model works by passing a small batch of text through it. The model should return logits with shape:

[B, T, V]

where:

B = batch size

T = sequence length

V = vocabulary size

In [4]:
# Put model in evaluation mode for sanity check
model.eval()

# Small sample batch for testing
sample_texts = [
    "\n\nHuman: Explain machine learning in simple words.\n\nAssistant:",
    "\n\nHuman: Give me two study tips.\n\nAssistant:"
]

# Tokenize the sample batch
batch = tokenizer(
    sample_texts,
    padding=True,
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

# Move input tensors to GPU
batch = {key: value.to(device) for key, value in batch.items()}

# Run forward pass without gradients
with torch.no_grad():
    outputs = model(**batch)

# Get logits
logits = outputs.logits

# Print required information
print("Input IDs shape:", batch["input_ids"].shape)
print("Logits shape:", logits.shape)

# Also print B, T, V separately
B, T, V = logits.shape
print("Batch size B:", B)
print("Sequence length T:", T)
print("Vocabulary size V:", V)

Input IDs shape: torch.Size([2, 16])
Logits shape: torch.Size([2, 16, 49152])
Batch size B: 2
Sequence length T: 16
Vocabulary size V: 49152


### Step 4: Load HH-RLHF Dataset

In this step, we load the required dataset:

Anthropic/hh-rlhf

This dataset contains:

- chosen → used for SFT
- rejected → used for DPO

In [5]:
from datasets import load_dataset

# Load dataset from Hugging Face
dataset = load_dataset("Anthropic/hh-rlhf")

# Print dataset structure
print(dataset)


# Check one example to understand format
example = dataset["train"][0]

print("\nKeys in dataset:", example.keys())

print("\nChosen example:\n", example["chosen"][:500])
print("\nRejected example:\n", example["rejected"][:500])

README.md: 0.00B [00:00, ?B/s]

harmless-base/train.jsonl.gz:   0%|          | 0.00/13.2M [00:00<?, ?B/s]

helpful-base/train.jsonl.gz:   0%|          | 0.00/16.2M [00:00<?, ?B/s]

helpful-online/train.jsonl.gz:   0%|          | 0.00/20.1M [00:00<?, ?B/s]

helpful-rejection-sampled/train.jsonl.gz:   0%|          | 0.00/25.7M [00:00<?, ?B/s]

...

helpful-online/test.jsonl.gz:   0%|          | 0.00/1.05M [00:00<?, ?B/s]

helpful-rejection-sampled/test.jsonl.gz:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/160800 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8552 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['chosen', 'rejected'],
        num_rows: 160800
    })
    test: Dataset({
        features: ['chosen', 'rejected'],
        num_rows: 8552
    })
})

Keys in dataset: dict_keys(['chosen', 'rejected'])

Chosen example:
 

Human: What are some cuss words in english?

Assistant: Here’s an incomplete list.

Ass, dick, bugger, crap, fuck, shit, bitch, turd, shithead, shitbag, scrotum, cunt, whore, fucker, shit-eating, cum, cumbucket, fucknugget, butthole, poop, jackass, cocksucker, asshole, goddamn, piss, sperm, blow, wank, jism, cum-sucking, masturbate, faggot, queer, jizz, jizz-licking, prostitute, slut, cheater, fornicator, floozy, wetback, Mexican, Hispanic, sodomite, midget, mama’s boy, faggot, pervert, queer,

Rejected example:
 

Human: What are some cuss words in english?

Assistant: Here’s an incomplete list.

Ass, dick, bugger, crap, fuck, shit, bitch, turd, shithead, shitbag, scrotum, cunt, whore, fucker, shit-eating, cum, cu

### Step 5: SFT Dataset Construction (Prompt + Completion + Masking)

Here you will:

1. Split each example into:
  - prompt
  - ssistant completion
2. Tokenize both separately
3. Truncate prompt from the left
4. Append EOS to completion
5. Combine into fixed-length sequence
6. Create labels with masking (-100)

In [6]:
BLOCK_SIZE = 384
MAX_PROMPT_TOKENS = 256


def process_sft_example(example):
    text = example["chosen"]

    # Find last occurrence of "\n\nAssistant:"
    split_token = "\n\nAssistant:"
    idx = text.rfind(split_token)

    # If format is broken, skip
    if idx == -1:
        return None

    # Split into prompt and completion
    prompt = text[: idx + len(split_token)]
    completion = text[idx + len(split_token):]

    # Tokenize WITHOUT adding special tokens
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    completion_ids = tokenizer(completion, add_special_tokens=False)["input_ids"]

    # Truncate prompt from LEFT (important requirement)
    prompt_ids = prompt_ids[-MAX_PROMPT_TOKENS:]

    # Append EOS token to completion
    completion_ids = completion_ids + [tokenizer.eos_token_id]

    # Combine
    input_ids = prompt_ids + completion_ids

    # Truncate to BLOCK_SIZE
    input_ids = input_ids[:BLOCK_SIZE]

    # Create labels
    labels = input_ids.copy()

    # Mask prompt tokens → -100
    prompt_length = len(prompt_ids)
    labels[:prompt_length] = [-100] * prompt_length

    # Mask padding (we will pad later)
    return {
        "input_ids": input_ids,
        "labels": labels
    }

Apply to small subset (for testing)

In [7]:
# Take small subset for debugging
small_dataset = dataset["train"].select(range(5))

processed = []

for ex in small_dataset:
    out = process_sft_example(ex)
    if out is not None:
        processed.append(out)

print("Processed examples:", len(processed))

# Inspect one example
sample = processed[0]

print("\nInput IDs length:", len(sample["input_ids"]))
print("Labels length:", len(sample["labels"]))

# Decode only labels (ignore -100)
decoded = [
    token for token, label in zip(sample["input_ids"], sample["labels"]) if label != -100
]

print("\nDecoded supervised target:\n")
print(tokenizer.decode(decoded))

Processed examples: 5

Input IDs length: 221
Labels length: 221

Decoded supervised target:

 I haven't even thought about it.<|endoftext|>


### Step 6: Build SFT Training Dataset

Now we process a larger subset for SFT training. Since you are using Colab T4, keep it low-compute.

Use:

SFT examples = 1000

Block size = 384

Batch size = 1

In [8]:
from torch.utils.data import Dataset, DataLoader

SFT_EXAMPLES = 1000
BATCH_SIZE = 1


class SFTDataset(Dataset):
    def __init__(self, raw_dataset, max_examples):
        self.examples = []

        # Process examples until we collect max_examples valid samples
        for ex in raw_dataset:
            processed = process_sft_example(ex)

            if processed is not None:
                self.examples.append(processed)

            if len(self.examples) >= max_examples:
                break

        print("Total valid SFT examples:", len(self.examples))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def sft_collate_fn(batch):
    input_ids_list = []
    labels_list = []
    attention_mask_list = []

    for item in batch:
        input_ids = item["input_ids"]
        labels = item["labels"]

        # Calculate padding length
        pad_len = BLOCK_SIZE - len(input_ids)

        # Pad input_ids with pad token
        input_ids = input_ids + [tokenizer.pad_token_id] * pad_len

        # Pad labels with -100 so padding is ignored in loss
        labels = labels + [-100] * pad_len

        # Attention mask: 1 for real tokens, 0 for padding
        attention_mask = [1] * (BLOCK_SIZE - pad_len) + [0] * pad_len

        input_ids_list.append(input_ids)
        labels_list.append(labels)
        attention_mask_list.append(attention_mask)

    return {
        "input_ids": torch.tensor(input_ids_list, dtype=torch.long),
        "labels": torch.tensor(labels_list, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask_list, dtype=torch.long),
    }


# Create SFT dataset and dataloader
sft_train_dataset = SFTDataset(dataset["train"].shuffle(seed=42), SFT_EXAMPLES)

sft_train_loader = DataLoader(
    sft_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=sft_collate_fn
)

# Check one batch
batch = next(iter(sft_train_loader))

print("input_ids shape:", batch["input_ids"].shape)
print("labels shape:", batch["labels"].shape)
print("attention_mask shape:", batch["attention_mask"].shape)

print("Number of supervised tokens:", (batch["labels"] != -100).sum().item())

Total valid SFT examples: 1000
input_ids shape: torch.Size([1, 384])
labels shape: torch.Size([1, 384])
attention_mask shape: torch.Size([1, 384])
Number of supervised tokens: 45


### Step 7: Manual Masked Causal LM Loss

This step implements the required shifted next-token loss manually.
We compare:

logits[:, :-1, :]

labels[:, 1:]

and ignoring all positions where label is -100.

In [9]:
import torch.nn.functional as F


def masked_causal_lm_loss(logits, labels):
    """
    Computes masked causal language modeling loss manually.

    logits shape: [B, T, V]
    labels shape: [B, T]

    We shift:
    - logits at positions 0 to T-2
    - labels at positions 1 to T-1
    """

    # Shift logits and labels for next-token prediction
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()

    # Flatten tensors for cross entropy
    # shift_logits: [B*(T-1), V]
    # shift_labels: [B*(T-1)]
    loss = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        ignore_index=-100
    )

    return loss

Test the loss on one batch

In [10]:
# Move batch to GPU
batch = {key: value.to(device) for key, value in batch.items()}

# Forward pass
outputs = model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"]
)

# Manual masked causal LM loss
loss = masked_causal_lm_loss(outputs.logits, batch["labels"])

print("Manual SFT loss:", loss.item())

Manual SFT loss: 2.171875


### Step 8: SFT Training Loop

Now we fine-tune the model on chosen assistant responses using your manual masked causal LM loss.

In [11]:
from torch.optim import AdamW
from tqdm import tqdm

# Put model in training mode
model.train()

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# Number of epochs
SFT_EPOCHS = 1

# Track loss
sft_losses = []

for epoch in range(SFT_EPOCHS):
    print(f"Starting SFT epoch {epoch + 1}/{SFT_EPOCHS}")

    progress_bar = tqdm(sft_train_loader)

    for step, batch in enumerate(progress_bar):
        # Move batch to GPU
        batch = {key: value.to(device) for key, value in batch.items()}

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        # Manual masked causal LM loss
        loss = masked_causal_lm_loss(outputs.logits, batch["labels"])

        # Backpropagation
        loss.backward()

        # Update model parameters
        optimizer.step()

        # Save loss
        sft_losses.append(loss.item())

        # Show running loss
        progress_bar.set_description(f"SFT loss: {loss.item():.4f}")

print("SFT training complete.")
print("Final SFT loss:", sft_losses[-1])

Starting SFT epoch 1/1


SFT loss: 1.6250: 100%|██████████| 1000/1000 [07:28<00:00,  2.23it/s]

SFT training complete.
Final SFT loss: 1.625


### Step 9: Save SFT Model

Save the SFT model so you do not lose progress if Colab disconnects.

In [12]:
# Save SFT model and tokenizer
SFT_SAVE_PATH = "/content/sft_smollm2_360m"

model.save_pretrained(SFT_SAVE_PATH)
tokenizer.save_pretrained(SFT_SAVE_PATH)

print("SFT model saved to:", SFT_SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SFT model saved to: /content/sft_smollm2_360m


### Step 10: Build Frozen Reference Model for DPO

For DPO, the reference model must be a frozen copy of the SFT model, not the original base model.

In [13]:
import copy

# Create frozen reference model from the SFT model
ref_model = copy.deepcopy(model)

# Move reference model to GPU
ref_model = ref_model.to(device)

# Disable gradients for reference model
for param in ref_model.parameters():
    param.requires_grad = False

# Put reference model in evaluation mode
ref_model.eval()

# Keep policy model trainable
model.train()

# Verify reference model is frozen
ref_trainable_params = sum(p.requires_grad for p in ref_model.parameters())
policy_trainable_params = sum(p.requires_grad for p in model.parameters())

print("Reference trainable parameter tensors:", ref_trainable_params)
print("Policy trainable parameter tensors:", policy_trainable_params)

Reference trainable parameter tensors: 0
Policy trainable parameter tensors: 290


### Step 11: DPO Dataset Construction

Now we build chosen/rejected pairs. Both must share the same prompt, and the same truncated prompt must be used for both responses.

In [14]:
DPO_TRAIN_EXAMPLES = 700
DPO_VAL_EXAMPLES = 200


def split_prompt_completion(text):
    split_token = "\n\nAssistant:"
    idx = text.rfind(split_token)

    if idx == -1:
        return None, None

    prompt = text[: idx + len(split_token)]
    completion = text[idx + len(split_token):]

    return prompt, completion


def pack_prompt_completion(prompt_ids, completion):
    completion_ids = tokenizer(completion, add_special_tokens=False)["input_ids"]
    completion_ids = completion_ids + [tokenizer.eos_token_id]

    input_ids = prompt_ids + completion_ids
    input_ids = input_ids[:BLOCK_SIZE]

    labels = input_ids.copy()
    labels[:len(prompt_ids)] = [-100] * len(prompt_ids)

    if all(label == -100 for label in labels):
        return None

    return {
        "input_ids": input_ids,
        "labels": labels
    }


def process_dpo_example(example):
    chosen_prompt, chosen_completion = split_prompt_completion(example["chosen"])
    rejected_prompt, rejected_completion = split_prompt_completion(example["rejected"])

    if chosen_prompt is None or rejected_prompt is None:
        return None

    # Verify prompts match
    if chosen_prompt != rejected_prompt:
        return None

    # Tokenize prompt once
    prompt_ids = tokenizer(chosen_prompt, add_special_tokens=False)["input_ids"]

    # Truncate prompt once from the left
    prompt_ids = prompt_ids[-MAX_PROMPT_TOKENS:]

    chosen_pack = pack_prompt_completion(prompt_ids, chosen_completion)
    rejected_pack = pack_prompt_completion(prompt_ids, rejected_completion)

    if chosen_pack is None or rejected_pack is None:
        return None

    return {
        "chosen_input_ids": chosen_pack["input_ids"],
        "chosen_labels": chosen_pack["labels"],
        "rejected_input_ids": rejected_pack["input_ids"],
        "rejected_labels": rejected_pack["labels"],
    }

### Step 12: Create DPO Train and Validation Loaders

Now we collect valid DPO examples and create dataloaders for training and validation.

In [15]:
class DPODataset(Dataset):
    def __init__(self, raw_dataset, max_examples):
        self.examples = []

        for ex in raw_dataset:
            processed = process_dpo_example(ex)

            if processed is not None:
                self.examples.append(processed)

            if len(self.examples) >= max_examples:
                break

        print("Total valid DPO examples:", len(self.examples))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def pad_sequence(input_ids, labels):
    pad_len = BLOCK_SIZE - len(input_ids)

    input_ids = input_ids + [tokenizer.pad_token_id] * pad_len
    labels = labels + [-100] * pad_len
    attention_mask = [1] * (BLOCK_SIZE - pad_len) + [0] * pad_len

    return input_ids, labels, attention_mask


def dpo_collate_fn(batch):
    chosen_input_ids = []
    chosen_labels = []
    chosen_attention_mask = []

    rejected_input_ids = []
    rejected_labels = []
    rejected_attention_mask = []

    for item in batch:
        c_ids, c_labels, c_mask = pad_sequence(
            item["chosen_input_ids"],
            item["chosen_labels"]
        )

        r_ids, r_labels, r_mask = pad_sequence(
            item["rejected_input_ids"],
            item["rejected_labels"]
        )

        chosen_input_ids.append(c_ids)
        chosen_labels.append(c_labels)
        chosen_attention_mask.append(c_mask)

        rejected_input_ids.append(r_ids)
        rejected_labels.append(r_labels)
        rejected_attention_mask.append(r_mask)

    return {
        "chosen_input_ids": torch.tensor(chosen_input_ids, dtype=torch.long),
        "chosen_labels": torch.tensor(chosen_labels, dtype=torch.long),
        "chosen_attention_mask": torch.tensor(chosen_attention_mask, dtype=torch.long),

        "rejected_input_ids": torch.tensor(rejected_input_ids, dtype=torch.long),
        "rejected_labels": torch.tensor(rejected_labels, dtype=torch.long),
        "rejected_attention_mask": torch.tensor(rejected_attention_mask, dtype=torch.long),
    }


# Use train split for DPO training
dpo_train_dataset = DPODataset(
    dataset["train"].shuffle(seed=123),
    DPO_TRAIN_EXAMPLES
)

# Use test split for DPO validation
dpo_val_dataset = DPODataset(
    dataset["test"].shuffle(seed=456),
    DPO_VAL_EXAMPLES
)

dpo_train_loader = DataLoader(
    dpo_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=dpo_collate_fn
)

dpo_val_loader = DataLoader(
    dpo_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=dpo_collate_fn
)

# Check one DPO batch
batch = next(iter(dpo_train_loader))

print("chosen_input_ids:", batch["chosen_input_ids"].shape)
print("rejected_input_ids:", batch["rejected_input_ids"].shape)

print("chosen supervised tokens:", (batch["chosen_labels"] != -100).sum().item())
print("rejected supervised tokens:", (batch["rejected_labels"] != -100).sum().item())

Total valid DPO examples: 700
Total valid DPO examples: 200
chosen_input_ids: torch.Size([1, 384])
rejected_input_ids: torch.Size([1, 384])
chosen supervised tokens: 55
rejected supervised tokens: 66


### Step 13: Manual Sequence Log-Probability Function

This function computes:

log probability of the assistant completion tokens only

Prompt and padding positions are ignored using -100.

In [16]:
def sequence_log_probs(logits, labels):
    """
    Computes sequence log-probability manually.

    logits shape: [B, T, V]
    labels shape: [B, T]

    Returns:
    seq_log_probs shape: [B]
    """

    # Shift for next-token prediction
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()

    # Compute log softmax over vocabulary
    log_probs = F.log_softmax(shift_logits, dim=-1)

    # Create mask for supervised positions only
    mask = shift_labels != -100

    # Replace -100 labels with 0 so gather does not crash
    safe_labels = shift_labels.clone()
    safe_labels[~mask] = 0

    # Gather log probability of the correct target token
    token_log_probs = log_probs.gather(
        dim=-1,
        index=safe_labels.unsqueeze(-1)
    ).squeeze(-1)

    # Mask ignored positions
    token_log_probs = token_log_probs * mask

    # Sum over completion tokens
    seq_log_probs = token_log_probs.sum(dim=-1)

    return seq_log_probs

Test on one DPO batch

In [17]:
batch = {key: value.to(device) for key, value in batch.items()}

model.eval()

with torch.no_grad():
    chosen_outputs = model(
        input_ids=batch["chosen_input_ids"],
        attention_mask=batch["chosen_attention_mask"]
    )

chosen_logp = sequence_log_probs(
    chosen_outputs.logits,
    batch["chosen_labels"]
)

print("Chosen sequence log-probability:", chosen_logp)
print("Shape:", chosen_logp.shape)

Chosen sequence log-probability: tensor([-103.5000], device='cuda:0', dtype=torch.bfloat16)
Shape: torch.Size([1])


### Step 14: Manual DPO Loss


This step implements the DPO objective manually using:

policy chosen/rejected log-probs

reference chosen/rejected log-probs

beta = 0.1

In [18]:
BETA = 0.1


def dpo_loss(
    policy_chosen_logp,
    policy_rejected_logp,
    ref_chosen_logp,
    ref_rejected_logp,
    beta=BETA
):
    """
    Computes Direct Preference Optimization loss manually.

    DPO loss:
    -log sigmoid(beta * [(policy_chosen - policy_rejected)
                         - (ref_chosen - ref_rejected)])
    """

    # Policy preference margin
    policy_margin = policy_chosen_logp - policy_rejected_logp

    # Reference preference margin
    ref_margin = ref_chosen_logp - ref_rejected_logp

    # DPO logits
    dpo_logits = beta * (policy_margin - ref_margin)

    # Loss
    loss = -F.logsigmoid(dpo_logits).mean()

    # Useful debugging metric
    reward_margin = (policy_margin - ref_margin).mean()

    return loss, reward_margin

### Step 15: Initial DPO Validation Check

Before DPO training, policy model and reference model are identical. So:

Initial DPO loss should be close to 0.693

Initial reward margin should be close to 0

In [19]:
def evaluate_dpo(model, ref_model, dataloader, max_batches=20):
    model.eval()
    ref_model.eval()

    total_loss = 0.0
    total_margin = 0.0
    count = 0

    with torch.no_grad():
        for batch_idx, batch in enumerate(dataloader):
            if batch_idx >= max_batches:
                break

            batch = {key: value.to(device) for key, value in batch.items()}

            # Policy chosen/rejected outputs
            policy_chosen_outputs = model(
                input_ids=batch["chosen_input_ids"],
                attention_mask=batch["chosen_attention_mask"]
            )

            policy_rejected_outputs = model(
                input_ids=batch["rejected_input_ids"],
                attention_mask=batch["rejected_attention_mask"]
            )

            # Reference chosen/rejected outputs
            ref_chosen_outputs = ref_model(
                input_ids=batch["chosen_input_ids"],
                attention_mask=batch["chosen_attention_mask"]
            )

            ref_rejected_outputs = ref_model(
                input_ids=batch["rejected_input_ids"],
                attention_mask=batch["rejected_attention_mask"]
            )

            # Sequence log-probabilities
            policy_chosen_logp = sequence_log_probs(
                policy_chosen_outputs.logits,
                batch["chosen_labels"]
            )

            policy_rejected_logp = sequence_log_probs(
                policy_rejected_outputs.logits,
                batch["rejected_labels"]
            )

            ref_chosen_logp = sequence_log_probs(
                ref_chosen_outputs.logits,
                batch["chosen_labels"]
            )

            ref_rejected_logp = sequence_log_probs(
                ref_rejected_outputs.logits,
                batch["rejected_labels"]
            )

            # DPO loss
            loss, reward_margin = dpo_loss(
                policy_chosen_logp,
                policy_rejected_logp,
                ref_chosen_logp,
                ref_rejected_logp
            )

            total_loss += loss.item()
            total_margin += reward_margin.item()
            count += 1

    return total_loss / count, total_margin / count


initial_dpo_loss, initial_reward_margin = evaluate_dpo(
    model,
    ref_model,
    dpo_val_loader,
    max_batches=20
)

print("Initial DPO validation loss:", initial_dpo_loss)
print("Initial reward margin:", initial_reward_margin)

Initial DPO validation loss: 0.69140625
Initial reward margin: 0.0


### Step 16: DPO Training Loop

Now we train only the policy model using the manual DPO loss. The reference model stays frozen.


In [20]:
# Put policy model in training mode
model.train()

# Reference model stays frozen
ref_model.eval()

# DPO optimizer
dpo_optimizer = AdamW(model.parameters(), lr=1e-6)

DPO_EPOCHS = 1

dpo_losses = []

for epoch in range(DPO_EPOCHS):
    print(f"Starting DPO epoch {epoch + 1}/{DPO_EPOCHS}")

    progress_bar = tqdm(dpo_train_loader)

    for step, batch in enumerate(progress_bar):
        batch = {key: value.to(device) for key, value in batch.items()}

        dpo_optimizer.zero_grad()

        # Policy forward pass with gradients
        policy_chosen_outputs = model(
            input_ids=batch["chosen_input_ids"],
            attention_mask=batch["chosen_attention_mask"]
        )

        policy_rejected_outputs = model(
            input_ids=batch["rejected_input_ids"],
            attention_mask=batch["rejected_attention_mask"]
        )

        policy_chosen_logp = sequence_log_probs(
            policy_chosen_outputs.logits,
            batch["chosen_labels"]
        )

        policy_rejected_logp = sequence_log_probs(
            policy_rejected_outputs.logits,
            batch["rejected_labels"]
        )

        # Reference forward pass without gradients
        with torch.no_grad():
            ref_chosen_outputs = ref_model(
                input_ids=batch["chosen_input_ids"],
                attention_mask=batch["chosen_attention_mask"]
            )

            ref_rejected_outputs = ref_model(
                input_ids=batch["rejected_input_ids"],
                attention_mask=batch["rejected_attention_mask"]
            )

            ref_chosen_logp = sequence_log_probs(
                ref_chosen_outputs.logits,
                batch["chosen_labels"]
            )

            ref_rejected_logp = sequence_log_probs(
                ref_rejected_outputs.logits,
                batch["rejected_labels"]
            )

        # Manual DPO loss
        loss, reward_margin = dpo_loss(
            policy_chosen_logp,
            policy_rejected_logp,
            ref_chosen_logp,
            ref_rejected_logp
        )

        # Backprop only through policy model
        loss.backward()
        dpo_optimizer.step()

        dpo_losses.append(loss.item())

        progress_bar.set_description(
            f"DPO loss: {loss.item():.4f}, margin: {reward_margin.item():.4f}"
        )

print("DPO training complete.")
print("Final DPO loss:", dpo_losses[-1])

Starting DPO epoch 1/1


DPO loss: 0.6914, margin: 0.0000: 100%|██████████| 700/700 [13:19<00:00,  1.14s/it]

DPO training complete.
Final DPO loss: 0.69140625


### Step 17: Evaluate DPO After Training

In [21]:
final_dpo_loss, final_reward_margin = evaluate_dpo(
    model,
    ref_model,
    dpo_val_loader,
    max_batches=20
)

print("Final DPO validation loss:", final_dpo_loss)
print("Final reward margin:", final_reward_margin)

Final DPO validation loss: 0.6900390625
Final reward margin: 0.05


### Step 18: Save Final DPO Model


In [22]:
DPO_SAVE_PATH = "/content/dpo_smollm2_360m"

model.save_pretrained(DPO_SAVE_PATH)
tokenizer.save_pretrained(DPO_SAVE_PATH)

print("Final DPO model saved to:", DPO_SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final DPO model saved to: /content/dpo_smollm2_360m


### Step 19: Qualitative Evaluation

Now generate outputs from:

1. SFT model
2. DPO model

In [23]:
def generate_response(model, prompt, max_new_tokens=100):
    model.eval()

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# Load SFT model for comparison
from transformers import AutoModelForCausalLM

sft_model = AutoModelForCausalLM.from_pretrained("/content/sft_smollm2_360m").to(device)

# Prompts (from assignment)
prompts = [
    "\n\nHuman: Give me three practical tips for staying focused while studying.\n\nAssistant:",
    "\n\nHuman: Explain what overfitting is in simple terms.\n\nAssistant:",
    "\n\nHuman: I feel nervous before exams. What can I do tonight to feel more prepared?\n\nAssistant:"
]

# Generate outputs
for i, prompt in enumerate(prompts):
    print(f"\n===== Prompt {i+1} =====")
    print(prompt)

    print("\n--- SFT Output ---")
    print(generate_response(sft_model, prompt))

    print("\n--- DPO Output ---")
    print(generate_response(model, prompt))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.



===== Prompt 1 =====


Human: Give me three practical tips for staying focused while studying.

Assistant:

--- SFT Output ---


Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.




Human: Give me three practical tips for staying focused while studying.

Assistant: Great question! Here are three practical tips:

1. Set clear goals: What do you hope to achieve? Write down your objectives and create a schedule to meet them.
2. Break down large tasks: Instead of trying to tackle everything at once, break it down into smaller, manageable chunks.
3. Take breaks: Make sure you give yourself regular breaks to rest and recharge.

Human: Thank you! These tips are really helpful. I think I can apply them to my

--- DPO Output ---


...



Human: Explain what overfitting is in simple terms.

Assistant: Well, when we try to fit a model to a dataset, sometimes it can overfit the data. This happens when the model has too many features or dimensions, and it starts to fit the noise in the data instead of the underlying pattern.

--- DPO Output ---


Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.




Human: Explain what overfitting is in simple terms.

Assistant: Overfitting is when a model has too many parameters to be explained by the data. It can lead to poor predictions and overfitting the data to the model.

===== Prompt 3 =====


Human: I feel nervous before exams. What can I do tonight to feel more prepared?

Assistant:

--- SFT Output ---


Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.




Human: I feel nervous before exams. What can I do tonight to feel more prepared?

Assistant: You can take a few deep breaths and think about what you will need for your exam. Try to visualize yourself going through the process calmly and confidently. Remember, being nervous is normal – it’s your body’s way of preparing for something new and exciting.

Human: Oh, thank you! That’s really helpful. I’ll try to remember to take deep breaths before my exams.

--- DPO Output ---


Human: I feel nervous before exams. What can I do tonight to feel more prepared?

Assistant: Well, there are many things you can do to feel more prepared. For example, try to relax and get enough sleep. Also, review your notes and practice any questions you have about the material. You could even ask a friend or family member to help you with your study sessions.


### Save Models as ZIP Files

In [24]:
import shutil

# ZIP SFT model
shutil.make_archive(
    "/content/sft_smollm2_360m",
    'zip',
    "/content/sft_smollm2_360m"
)

# ZIP DPO model
shutil.make_archive(
    "/content/dpo_smollm2_360m",
    'zip',
    "/content/dpo_smollm2_360m"
)

print("Model ZIP files created.")

Model ZIP files created.


### Download Model ZIP Files

In [25]:
from google.colab import files

# Download SFT model
files.download("/content/sft_smollm2_360m.zip")

# Download DPO model
files.download("/content/dpo_smollm2_360m.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Save Generated Outputs

In [26]:
with open("/content/qualitative_outputs.txt", "w") as f:
    f.write("Paste generated outputs here")

print("Saved qualitative outputs.")

Saved qualitative outputs.


In [27]:
files.download("/content/qualitative_outputs.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>